# Sequence modeling: Transformers

In [ ]:
from __future__ import print_function
import tensorflow as tf
import os, json, re
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from keras.preprocessing.text import Tokenizer # If you work on colab you might need to use from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import pandas as pd

### Loading the Reuters newswire classification dataset

The reuters newswire classification dataset is a dataset of 11,228 newswires from Reuters, labeled over 46 topics. More information about the dataset and how to use it can be found here:
https://keras.io/api/datasets/reuters/

Try to build a new model dealing with this new dataset.
Be carefull that this will imply some changes in your code as it is not a binary classification problem anymore!

In [ ]:
maxlen = 30 # <---- investigate this length (decrease and increase)
max_features = 10000 # investigate this number (e.g., decrease to a very low number)

reuters = tf.keras.datasets.reuters
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=max_features)

num_classes = np.max(y_train) + 1
print("Number of classes: ", num_classes)

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

print("Training set size: ", np.shape(x_train))
print("Test set size: ", np.shape(x_test))

In [ ]:
word_to_index = reuters.get_word_index()
vocab_size = len(word_to_index)
print('Vocab size : ', vocab_size)

mostcommon = 10 # <---- investigate this number

words_freq_list =words_freq_list = []
[]
for (k,v) in reuters.get_word_index().items():
    words_freq_list.append((k,v))

sorted_list = sorted(words_freq_list, key=lambda x: x[1])

print(mostcommon, " most common words in reuters: \n")
print(sorted_list[0:mostcommon])

### Using Transformers

<img src="https://machinelearningmastery.com/wp-content/uploads/2021/08/attention_research_1-768x1082.png" width="250px" align="right"><br>

One main drawback about RNNs is their capacity to remember long-term dependencies. To alleviate this problem different models have been proposed, like Long Short Term Memories (LSTM) and Transformers.<br>
*Transformers* is one of the best available model nowadays to deal with different kind of data (text, images..) and obtain state of the art results.

The key component in the Transformer architecture is the Attention layer, that helps the encoder look at other words in the input sentence as it encodes a specific word. The attention mechanism, in theory, and given enough compute resources, have a wider window to reference from, therefore being capable of using the **entire context** of the text.

In [ ]:
# Define the Transformer block in TensorFlow

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [tf.keras.layers.Dense(ff_dim, activation="relu"), tf.keras.layers.Dense(embed_dim),]
        )
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
# Define the input embeddings and the positional encoding

class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim) # What does this layer do?
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [ ]:
# Define an architecture with the representation part employing the Transformer, followed by a classifier

embed_dim = 32  # Embedding size for each token, try and change it
num_heads = 2  # Number of attention heads, try and change it
ff_dim = 32  # Hidden layer size in feed forward network inside transformer

# The following is a possible model, reason on what you can change on the structure

model = tf.keras.Sequential()
model.add(TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)) # use the 
model.add(TransformerBlock(embed_dim, num_heads, ff_dim))
model.add(tf.keras.layers.GlobalAveragePooling1D()) 
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Dense(20, activation="relu"))
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Dense(46, activation="softmax"))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])
input_shape=(None, maxlen)
model.build(input_shape)
model.summary()


In [ ]:
# ... train the model here, remember to use a validation split
history = model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.1
)

In [ ]:
def plot_history(history):
    # Plot training & validation accuracy values
    plt.plot(history.history['acc'])
    plt.plot(history.history['val_acc'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Val'], loc='upper left')
    plt.show()

    # Plot training & validation loss values
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Val'], loc='upper left')
    plt.show()

In [ ]:
plot_history(history)

In [ ]:
# ... evaluate the trained model on the test set ...
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")